In [1]:
import sys
import os

os.chdir("/home/duoc/workflow")
sys.path.insert(0, "/home/duoc/workflow") 

import shutil
from pathlib import Path
from app.rag.service import RAG

shutil.rmtree("./rag_storage", ignore_errors=True)
Path("./rag_storage").mkdir()

rag = RAG()  # khởi tạo lại với store trống

/home/duoc/workflow/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
INFO:rag:[config] device=cpu dtype=torch.float32
INFO:rag:[tokenizer] fallback regex
INFO:sentence_transformers.base.model:No device provided, using cpu
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
IN

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/main "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/discussions?p=0 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/refs%2Fpr%2F130 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bg

In [2]:
import ipywidgets as w
from IPython.display import display
import asyncio
from pathlib import Path
from app.rag.service import RAG
from app.rag.loader import DocumentLoader

rag, loader = RAG(), DocumentLoader()

upload = w.FileUpload(accept=".pdf,.docx,.txt,.md,.csv", multiple=False)
btn = w.Button(description="⬆️ Ingest", button_style="primary")

progress = w.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Tiến trình:"
)

status = w.HTML("<b>🟡 Chờ chọn file...</b>")
log = w.HTML("")

async def on_click(_):
    if not upload.value:
        status.value = "<b style='color:red'>⚠️ Chọn file trước</b>"
        return

    try:
        file = upload.value[0] if isinstance(upload.value, (list, tuple)) else list(upload.value.values())[0]
        fname = file.name if hasattr(file, "name") else file["metadata"]["name"]
        fdata = file.data if hasattr(file, "data") else file["content"]

        progress.bar_style = "info"
        progress.value = 5
        status.value = f"<b>📂 {fname}</b>"

        tmp = Path(f"/tmp/{fname}")
        tmp.write_bytes(fdata)

        progress.value = 20
        status.value = "<b>📖 Đang đọc file...</b>"

        docs = loader.load_file(str(tmp))
        if not docs:
            raise Exception("Không đọc được nội dung file")

        text = docs[0].text

        progress.value = 40
        status.value = f"<b>📄 {len(text):,} ký tự</b>"

        progress.value = 60
        status.value = "<b>🧠 Đang ingest vào Vector DB...</b>"

        await asyncio.wait_for(
            rag.add(
                text,
                source_id=fname,
                document_type="upload"
            ),
            timeout=300
        )

        progress.value = 100
        progress.bar_style = "success"

        status.value = "<b style='color:green'>✅ Hoàn tất</b>"

        log.value = f"""
        <div style="padding:10px;border-left:4px solid green;margin-top:10px">
            <b>File:</b> {fname}<br>
            <b>Ký tự:</b> {len(text):,}<br>
            <b>Trạng thái:</b> Đã lưu vào Vector DB
        </div>
        """

        tmp.unlink(missing_ok=True)

    except asyncio.TimeoutError:
        progress.bar_style = "danger"
        status.value = "<b style='color:red'>❌ Timeout sau 300 giây</b>"

    except Exception as e:
        progress.bar_style = "danger"
        status.value = f"<b style='color:red'>❌ {e}</b>"

btn.on_click(lambda b: asyncio.create_task(on_click(b)))

display(
    w.VBox([
        upload,
        btn,
        progress,
        status,
        log
    ])
)

INFO:sentence_transformers.base.model:No device provided, using cpu
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from BAAI/bge-m3.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BA

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/main "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/discussions?p=0 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/refs%2Fpr%2F130 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bg

In [3]:

TEST_QUERIES = [
    # Entity queries
    "Giám đốc điều hành là ai?",
    "CEO của công ty là ai?",
    "Người đại diện điều hành là ai?",
    "Trưởng bộ phận Giảng dạy là ai?",
    "Trưởng bộ phận Nội dung là ai?",
    "Trưởng bộ phận Kỹ thuật là ai?",
    "Giám đốc tài chính là ai?",
    
    # Code/ID queries
    "Mã nội bộ chương trình AI Foundation là gì?",
    "Mã nội bộ của chương trình Data Master là gì?",
    "Mã nội bộ chương trình Cloud Expert là gì?",
    "Mã nội bộ chương trình DevOps Pro là gì?",
    "Mã nội bộ chương trình Business Analyst là gì?",
    "Mã xác thực nội bộ là gì?",
    "Mật danh trung tâm điều phối là gì?",
    "Khóa hệ thống dự phòng là gì?",
    "Mã doanh nghiệp nội bộ là gì?",
    
    # Numeric queries
    "Ngân sách dành cho phát triển giáo trình là bao nhiêu?",
    "Ngân sách marketing là bao nhiêu?",
    "Ngân sách đầu tư cơ sở vật chất là bao nhiêu?",
    "Tổng ngân sách năm 2026 là bao nhiêu?",
    "Chi phí vận hành dự kiến là bao nhiêu?",
    "Doanh thu quý I là bao nhiêu?",
    "Doanh thu quý II là bao nhiêu?",
    "Tổng số nhân sự là bao nhiêu?",
    "Bộ phận Giảng dạy có bao nhiêu giảng viên?",
    "Bộ phận Tư vấn có bao nhiêu người?",
    "Mục tiêu doanh thu 2027 là bao nhiêu?",
    "Tuyển mới thêm bao nhiêu người?",
    
    # Purpose queries
    "Cơ sở Kim Mã dùng để làm gì?",
    "Cơ sở Nguyễn Văn Linh dùng để làm gì?",
    "Cơ sở Lê Lợi dùng để làm gì?",
    "Cơ sở Trần Phú dùng để làm gì?",
    "Chương trình AI Foundation là gì?",
    "Chương trình Data Master là gì?",
    "Chương trình Cloud Expert là gì?",
    "Chương trình DevOps Pro là gì?",
    "Chương trình Business Analyst là gì?",
    
    # Location queries
    "Trụ sở chính ở đâu?",
    "Cơ sở Kim Mã đặt tại đâu?",
    "Cơ sở Nguyễn Văn Linh ở đâu?",
    "Cơ sở Lê Lợi đặt tại đâu?",
    "Cơ sở Trần Phú ở đâu?",
    "Cơ sở mới mở ở đâu?",
    
    # General queries
    "Tên chiến dịch tuyển sinh quý III là gì?",
    "Chương trình nào liên quan đến AI?",
    "Chương trình nào liên quan đến cloud?",
    "Cơ sở nào đào tạo lập trình viên AI?",
    "Cơ sở nào đào tạo online?",
    "Có bao nhiêu bộ phận?",
    "Quy định nghỉ phép như thế nào?",
    "Làm việc từ xa được bao nhiêu ngày?",
    "Dữ liệu học viên được mã hóa như thế nào?",
    "Kế hoạch năm 2027 có gì?",
    "Công ty hợp tác với bao nhiêu doanh nghiệp?",
    
    # Edge cases
    "Tên công ty là gì?",
    "Ngày thành lập công ty?",
    "Email CEO là gì?",
    "Có mấy cơ sở đào tạo?",
    "Có mấy chương trình đào tạo?",
    "Bản quyền giáo trình thuộc về ai?",
]




In [4]:
import os, time, psutil
import ipywidgets as w
from IPython.display import display, Markdown

from app.rag.service import RAG

rag = RAG()
proc = psutil.Process(os.getpid())

def metrics(elapsed, cpu, delta_rss, rss):
    display(w.HBox([
        w.Label(f"⏱ {elapsed:.3f}s"),
        w.Label(f"🖥 {cpu:.1f}%"),
        w.Label(f"📈 {delta_rss:+.1f}MB"),
        w.Label(f"🗂 {rss:.1f}MB"),
    ]))

def measure():
    proc.cpu_percent(None)
    return proc.memory_info().rss / 1024**2, time.perf_counter()

def read(m0, t0):
    elapsed = time.perf_counter() - t0
    cpu = proc.cpu_percent(None)
    rss = proc.memory_info().rss / 1024**2
    return elapsed, cpu, rss - m0, rss

display(Markdown("## 🔍 RAG Search Test"))

for i, q in enumerate(TEST_QUERIES, start=1):

    m0, t0 = measure()

    result = await rag.search(q, top_k=3)

    elapsed, cpu, delta_rss, rss = read(m0, t0)

    display(Markdown(f"### {i}. {q}"))
    display(Markdown(f"**Nguồn:** `{result.source}`"))

    if not result.chunks:
        print("Không có kết quả")
    else:
        for rank, chunk in enumerate(result.chunks, start=1):
            sid = chunk.meta.get("source_id", "?")
            txt = chunk.text[:200].replace("\n", " ")

            print(
                f"[{rank}] "
                f"score={chunk.score:.4f} "
                f"source={sid}"
            )
            print(txt)
            print("-" * 80)

    metrics(elapsed, cpu, delta_rss, rss)

display(Markdown("---\n## ⚡ Cache Test"))

t0 = time.perf_counter_ns()
await rag.search(TEST_QUERIES[0])
cold = time.perf_counter_ns() - t0

t0 = time.perf_counter_ns()
await rag.search(TEST_QUERIES[0])
cached = time.perf_counter_ns() - t0

display(w.HBox([
    w.Label(f"🥶 Cold: {cold/1e6:.2f}ms"),
    w.Label(f"✅ Cache: {cached/1e6:.2f}ms"),
    w.Label(f"🚀 Speedup: {cold/(cached+1):.0f}x"),
]))

display(Markdown("---\n## 📊 Store Stats"))

s = rag._store.stats()

display(w.HBox([
    w.Label(f"Chunks: {s['chunks']}"),
    w.Label(f"Hashes: {s['hashes']}"),
    w.Label(f"FAISS: {s['faiss_vectors']}"),
]))

INFO:sentence_transformers.base.model:No device provided, using cpu
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from BAAI/bge-m3.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BA

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/main "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/discussions?p=0 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/refs%2Fpr%2F130 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bg

## 🔍 RAG Search Test

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Không có kết quả


---
## ⚡ Cache Test

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

---
## 📊 Store Stats